# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Core workflow**
1. Ingest a **GMI** and an **SSMIS** granule → Parquet partitions on S3 + RDS metadata (two instruments so the overlap analytics in step 10 have something to compare)
2. Find intersecting data for a bounding box via STARE SIDs + RDS
3. Download intersecting Parquet partitions from S3
4. Reconstitute an HDF5 file (both S1 and S2 scans)
5. Compare the reconstituted structure with the original granule
6. Verify RDS metadata

**Temporal features** (temporal-stare-pods issues 01–06)
7. Temporal catalog — every chunk carries `[t_start, t_end]` + podcode
8. Period-filtered load — data-level `[t_start, t_end]` overlap
9. VCF temporal roll-up — union range per pod, on the fly
10. Multi-instrument overlap analytics — the slide-8/9 rendezvous views

> The S3/RDS temporal loaders read the **shared** `PodsMetadata` catalog — every ingest in the RDS table, not only this demo's granule (filtered by instrument). That is the production query surface, so the temporal counts reflect the whole catalog, unlike the local demo's fresh isolated SQLite.

**Requires** `starepandas/.config` with AWS + RDS credentials. Sample granules default to the in-repo GMI + SSMIS; override with `STAREPODS_SAMPLE_GRANULE` / `STAREPODS_SAMPLE_GRANULE_SSMIS`.

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [2]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

## Configuration

Edit these paths and parameters before running.

In [3]:
import starepandas

# AWS + RDS credentials. Resolved relative to the installed package so it works
# from any cwd (the .config lives next to the starepandas package).
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(starepandas.__file__)), ".config")

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5",
    ),
)

# A second instrument (SSMIS) so the overlap analytics (step 10) span two
# instruments. The F18 granule (2025-01-05) is the closest in time to the GMI
# granule (2025-01-01) among the in-repo samples.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5",
    ),
)

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 10

# Bounding box filter — set to None to reconstitute the full granule
# (matching local_starepods_examples.ipynb), or e.g. (115, -30, 120, -25)
# to restrict to SW Australia / Perth.
BBOX = None   # full granule, no spatial filter — mirrors the local demo

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS    : {os.path.basename(SSMIS_GRANULE_FILE)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

Granule  : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
SSMIS    : 1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5
Datasets : ['GMI_S1', 'GMI_S2']
BBox     : None  (None = full granule)
S3 root  : s3://zarrpods/gmi-demo-parquet
Clean    : True


## Step 1 — Ingest granule → S3 Parquet + RDS

In [4]:
%%time
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
# Second instrument appends — clean_before_run=False so it does NOT wipe
# the GMI data just written to the same prefix.
ssmis_paths = demo.ingest_granules(
    data_path=SSMIS_GRANULE_FILE,
    instrument="SSMIS",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=False,
)
print(f"GMI  : stored {len(s3_paths)} dataset path(s)")
print(f"SSMIS: stored {len(ssmis_paths)} dataset path(s)")

# Granule basename — used as a substring filter on group_path. Note: as of
# the quaternary pod-code layout (2026-06-14) the S3 layout is FLAT and the
# granule basename is embedded in the chunk *filename*, bracketed by '-':
#   <S3_PREFIX>/<podcode>-<granule_basename>-<dataset>.parquet
# So the old startswith(S3_PREFIX + '/' + basename) scoping no longer matches.
# We use a substring match on the basename instead.
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_path_marker = f"-{granule_basename}-"   # matches the filename-embedded span

INFO:starepandas.ingest:clean_before_run=True → wiping s3://zarrpods/gmi-demo-parquet on S3 + RDS first


INFO:starepandas.ingest:clean_s3_prefix(s3://zarrpods/gmi-demo-parquet): deleted 2032 RDS row(s), 2032 S3 object(s)


INFO:starepandas.ingest:Ingesting GMI granules from /Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_pods_aws_parallel/tests/data/granules/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


...

  Progress: 300/380 partitions written...


  Progress: 350/380 partitions written...


INFO:starepandas.ingest:✓ Stored 1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5 → ['s3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet']


INFO:starepandas.ingest:Ingested 4 Parquet dataset(s)


✓ Inserted 380 metadata rows into RDS
✓ Finished writing 380 Parquet partitions to s3://zarrpods/gmi-demo-parquet
GMI  : stored 2 dataset path(s)
SSMIS: stored 4 dataset path(s)
CPU times: user 31.7 s, sys: 2.63 s, total: 34.3 s
Wall time: 3min 52s


## Step 2 — Find intersecting data via STARE SIDs

In [5]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule. Substring match on the basename, which the flat
    # pod-code layout embeds in the chunk filename (bracketed by '-').
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.contains(granule_path_marker, regex=False)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)


BBOX is None — Step 4 will reconstitute the full granule directly.


## Step 3 — Download intersecting Parquet partitions from S3

In [6]:
%%time
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

No intersecting partitions to download — Step 4 will read S3 directly.
CPU times: user 38 μs, sys: 11 μs, total: 49 μs
Wall time: 45.1 μs


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [7]:
%%time
# s3_prefix scope: with CLEAN_BEFORE_RUN=True the bucket only holds this
# granule's data, so passing the broad S3_PREFIX is correct and avoids the
# layout mismatch the old per-granule S3 prefix would create.
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=S3_PREFIX,
)
print(f"Written to: {recon_path}")


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over full granule (no spatial filter)


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over full granule (no spatial filter)


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/gmi_s3_reconstituted.h5


Written to: /tmp/gmi_s3_reconstituted.h5
CPU times: user 12.6 s, sys: 3.4 s, total: 16 s
Wall time: 1min 45s


## Step 5 — Structure comparison: reconstituted vs original

In [8]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_s3_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear       

## Step 6 — RDS metadata verification

In [9]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        # Flat pod-code layout: the basename is embedded in the chunk
        # filename, so use a LIKE substring match (not a startswith prefix).
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"%{granule_path_marker}%"),
        )
        rows = cur.fetchall()
    print(f"RDS scope: group_path contains '{granule_path_marker}'")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()


RDS scope: group_path contains '-1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B-'
  GMI_S1: 263 partition(s)
  GMI_S2: 251 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

`load_s3_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) from RDS — never the heavy `MetadataJson`. This reads the **shared** catalog filtered by instrument, so the counts include every GMI/SSMIS ingest in the table, not just this granule.

In [10]:
import pandas as pd
from starepandas.io.granules import (
    load_s3_metadata, load_s3_temporal_catalog, load_s3_vcf,
)
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table,
    pair_drilldown, pod_drilldown,
)

catalog = pd.concat(
    [load_s3_temporal_catalog(dataset_prefix='GMI'),
     load_s3_temporal_catalog(dataset_prefix='SSMIS')],
    ignore_index=True,
)
print(f"Thin catalog (GMI+SSMIS, catalog-wide): {len(catalog)} chunks across "
      f"{catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog (GMI+SSMIS, catalog-wide): 16257 chunks across 6 datasets


,chunks,first_start,last_end
Dataset,,,
GMI_S1,263,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739
GMI_S2,251,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739
SSMIS_S1,3951,2025-01-01 01:17:03.438,2025-01-06 00:07:25.237
SSMIS_S2,3947,2025-01-01 01:17:03.438,2025-01-06 00:07:25.237
SSMIS_S3,3907,2025-01-01 01:17:03.438,2025-01-06 00:07:25.237
SSMIS_S4,3938,2025-01-01 01:17:03.438,2025-01-06 00:07:25.237


,podcode,Dataset,t_start,t_end
0,q32203,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:45:32.515
1,q32202,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:45:32.515
2,q32231,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:58.766
3,q32221,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:49.391
4,q33012,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:49.391
5,q33032,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:44:40.015


## Step 8 — Period-filtered load

`load_s3_metadata(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period (the shared `_period_conditions` index-friendly rewrite — live EXPLAIN confirms a Bitmap Index Scan on `idx_pods_temporal`). A window bracketing the GMI window returns its chunks; a window days away returns none.

In [11]:
gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
gmi_start, gmi_end = gmi['t_start'].min(), gmi['t_end'].max()
match_period = (gmi_start - pd.Timedelta(hours=1), gmi_end + pd.Timedelta(hours=1))
miss_period  = (gmi_start - pd.Timedelta(days=10), gmi_start - pd.Timedelta(days=9))

hit  = load_s3_metadata(dataset_prefix='GMI', period=match_period)
miss = load_s3_metadata(dataset_prefix='GMI', period=miss_period)

print(f"GMI catalog window : [{gmi_start}, {gmi_end}]")
print(f"bracketing period  -> {len(hit)} chunks")
print(f"9-10 days earlier  -> {len(miss)} chunks")

GMI catalog window : [2025-01-01 03:43:47.516000, 2025-01-01 05:16:58.739000]
bracketing period  -> 514 chunks
9-10 days earlier  -> 0 chunks


## Step 9 — VCF temporal roll-up

`load_s3_vcf(level, ...)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` + child count — the on-the-fly temporal hierarchy ("Virtual Collection File"), nothing materialized.

In [12]:
vcf = load_s3_vcf(1, dataset_prefix='GMI')
print(f"{len(vcf)} level-1 VCF nodes for GMI (one per octant subtree)")
vcf

18 level-1 VCF nodes for GMI (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q01,2025-01-01 03:46:47.515,2025-01-01 04:01:04.385,64,0
1,q11,2025-01-01 03:59:17.511,2025-01-01 04:05:40.009,33,0
2,q12,2025-01-01 04:16:45.630,2025-01-01 04:21:19.379,18,0
3,q13,2025-01-01 04:04:11.884,2025-01-01 04:18:34.380,56,0
4,q30,2025-01-01 05:06:23.117,2025-01-01 05:07:24.991,2,0
5,q31,2025-01-01 03:46:36.265,2025-01-01 03:50:26.888,12,0
6,q32,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739,47,0
7,q33,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739,31,0
8,q40,2025-01-01 05:05:54.992,2025-01-01 05:08:21.241,7,0
9,q41,2025-01-01 04:48:43.747,2025-01-01 04:52:24.996,12,0


## Step 10 — Multi-instrument overlap analytics (slides 8/9)

`rendezvous_events` sweeps the catalog for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs aggregate that one events frame.

> **Note on the sample data.** The in-repo GMI (2025-01-01) and SSMIS (2025-01-05) passes are days apart and land in different pods, so a realistic Δt finds **no** rendezvous — the sweep is correct, the sample granules just don't co-locate. Swap in co-located GMI/SSMIS granules (via the env vars) and this real sweep lights up with no code change. The synthetic catalog below shows what the views look like when data *does* overlap.

In [13]:
dt = pd.Timedelta(minutes=30)
real_events = rendezvous_events(catalog, dt)
print(f"Rendezvous over the GMI+SSMIS catalog (dt={dt}): {len(real_events)} events")

Rendezvous over the GMI+SSMIS catalog (dt=0 days 00:30:00): 0 events


### Illustrative synthetic catalog (co-located passes)

Three pods: one where GMI + SSMIS + ATMS all pass within 30 min (a **trio**), one with a GMI + SSMIS **pair**, and one with GMI alone. Same shape a temporal-catalog loader returns.

In [14]:
m = lambda mins: pd.Timedelta(minutes=mins)
t = pd.Timestamp('2025-01-01 10:00')
demo_cat = pd.DataFrame(
    [   # pod q13011: GMI + SSMIS + ATMS within 30 min -> a trio
        ('q13011', 'GMI_S1',   t,         t + m(2)),
        ('q13011', 'SSMIS_S1', t + m(12), t + m(15)),
        ('q13011', 'ATMS_S1',  t + m(20), t + m(23)),
        # pod q13012: GMI + SSMIS -> a pair
        ('q13012', 'GMI_S1',   t + m(120), t + m(123)),
        ('q13012', 'SSMIS_S1', t + m(140), t + m(144)),
        # pod q13013: GMI alone -> no rendezvous
        ('q13013', 'GMI_S1',   t + m(300), t + m(303)),
    ],
    columns=['podcode', 'Dataset', 't_start', 't_end'],
)
ev = rendezvous_events(demo_cat, dt)
print(f"{len(ev)} events over {demo_cat['podcode'].nunique()} pods")

print('\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8):')
display(overlap_matrix(ev))

print('Per-pod n-way combination counts (slide 9):')
display(overlap_pod_table(ev))

print('GMI-SSMIS pair drill-down (pods + times):')
display(pair_drilldown(ev, 'GMI', 'SSMIS'))

print("Subtree drill-down under 'q1301' (rolls up q13011/12/13):")
display(pod_drilldown(ev, 'q1301'))

3 events over 3 pods

Instrument x instrument matrix — pods where A & B rendezvous (slide 8):


,ATMS,GMI,SSMIS
ATMS,0,1,1
GMI,1,0,2
SSMIS,1,2,0


Per-pod n-way combination counts (slide 9):


n_instruments,2,3
podcode,,
q13011,3,1
q13012,1,0


GMI-SSMIS pair drill-down (pods + times):


,podcode,frequency,times
0,q13011,1,[2025-01-01 10:12:00]
1,q13012,1,[2025-01-01 12:20:00]


Subtree drill-down under 'q1301' (rolls up q13011/12/13):


,instruments,n_instruments,frequency,times
0,"(ATMS, GMI)",2,1,[2025-01-01 10:20:00]
1,"(ATMS, SSMIS)",2,1,[2025-01-01 10:20:00]
2,"(GMI, SSMIS)",2,2,"[2025-01-01 10:12:00, 2025-01-01 12:20:00]"
3,"(ATMS, GMI, SSMIS)",3,1,[2025-01-01 10:20:00]
